# Dental Image Classifier — Google Colab (ResNet-18, transfer learning)

Notebook para treinamento do classificador de vistas intraorais odontológicas por **transfer learning**: um extrator de features ResNet-18 pré-treinado na ImageNet, mais um classificador linear treinado sobre essas features.

**Vistas classificadas:** frontal, inferior, superior, lateral direita, lateral esquerda

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taynaramos/dental-image-classifier/blob/main/notebooks/pytorch_resnet18_transfer_colab.ipynb)

> **Recomendado:** habilite GPU em *Ambiente de execução → Alterar tipo de ambiente de execução → GPU*.

## 1. Clonar o repositório

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/taynaramos/dental-image-classifier.git"
    BRANCH = "main"
    PROJECT_ROOT = Path("/content/dental-image-classifier")
    if not PROJECT_ROOT.exists():
        !git clone --branch {BRANCH} {REPO_URL} {PROJECT_ROOT}
else:
    # Permite executar este mesmo notebook localmente, a partir de notebooks/
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Projeto em: {PROJECT_ROOT}")

## 2. Download do dataset

O zip do dataset é baixado direto do Google Drive via `gdown` (link compartilhado "qualquer pessoa com o link") — **sem precisar montar o Drive nem autorizar acesso**. O arquivo é salvo no disco local da VM e extraído em `data/dataset`.

In [ ]:
import shutil
import zipfile

# ID do arquivo no Google Drive (link "qualquer pessoa com o link pode ver")
GDRIVE_FILE_ID = "11XFNZe6KpP2Jkhb6SkAOuYkl3rfEoOX9"

DATASET_ROOT = PROJECT_ROOT / "data" / "dataset"

if IN_COLAB and not DATASET_ROOT.exists():
    zip_local = Path("/content/Dataset_CINUFPE_Odontologico.zip")
    if not zip_local.exists():
        !gdown {GDRIVE_FILE_ID} -O {zip_local}

    print("Extraindo …")
    with zipfile.ZipFile(zip_local) as zf:
        zf.extractall(DATASET_ROOT)
    # Se o zip tiver uma única pasta raiz, desaninha para DATASET_ROOT
    conteudos = list(DATASET_ROOT.iterdir())
    if len(conteudos) == 1 and conteudos[0].is_dir():
        raiz_unica = conteudos[0]
        for filho in raiz_unica.iterdir():
            shutil.move(str(filho), DATASET_ROOT)
        raiz_unica.rmdir()

n_sujeitos = sum(1 for p in DATASET_ROOT.iterdir() if p.is_dir())
print(f"Dataset em : {DATASET_ROOT}")
print(f"Sujeitos   : {n_sujeitos}")

## 3. Importações e verificação da GPU

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from src.pytorch_resnet18_transfer.dataset import build_dataloaders, resolve_imagefolder_root
from src.pytorch_resnet18_transfer.model import DentalResNetTransfer
from src.pytorch_resnet18_transfer.predict import predict
from src.pytorch_resnet18_transfer.trainer import Trainer
from src.pytorch_resnet18_transfer.utils import get_device, save_checkpoint, set_seed

%matplotlib inline

device = get_device()
print(f"Dispositivo: {device}")
if device.type == "cuda":
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
else:
    print("AVISO: sem GPU — selecione um ambiente de execução com GPU para acelerar o treino.")

## 4. Configuração

`IMAGE_SIZE=224` é o tamanho esperado pela ResNet-18 pré-treinada na ImageNet. `BATCH_SIZE` e `NUM_WORKERS` maiores para aproveitar a GPU.

In [ ]:
MODEL_OUT = PROJECT_ROOT / "artifacts" / "resnet18_transfer_model.pth"

IMAGE_SIZE      = 224
BATCH_SIZE      = 64
NUM_WORKERS     = 2
FROZEN_EPOCHS   = 5    # Fase 1: extrator congelado, treina só o classificador
FINETUNE_EPOCHS = 15   # Fase 2: fine-tune da rede inteira
HEAD_LR         = 1e-3
FINETUNE_LR     = 1e-4
SEED            = 42

set_seed(SEED)
print(f"Modelo (out): {MODEL_OUT}")

## 5. Preparo do dataset e DataLoaders

In [ ]:
# Materializa (uma única vez) o dataset por sujeito em train/val/test por classe.
# A divisão é feita por sujeito, para não vazar dados do mesmo paciente entre conjuntos.
imagefolder_root = resolve_imagefolder_root(DATASET_ROOT, seed=SEED)

train_loader, val_loader, test_loader, classes = build_dataloaders(
    imagefolder_root,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
NUM_CLASSES = len(classes)

print(f"Classes ({NUM_CLASSES}): {classes}")
print(f"Batches — treino: {len(train_loader)}  val: {len(val_loader)}  teste: {len(test_loader)}")

## 6. Visualização de algumas imagens

In [ ]:
classes_preview = sorted(p.name for p in (imagefolder_root / "train").iterdir() if p.is_dir())

fig, axes = plt.subplots(1, len(classes_preview), figsize=(15, 3))
for ax, classe in zip(axes, classes_preview):
    exemplo = random.choice(list((imagefolder_root / "train" / classe).glob("*")))
    ax.imshow(Image.open(exemplo))
    ax.set_title(classe, fontsize=9)
    ax.axis('off')
fig.suptitle("Um exemplo aleatório por classe (conjunto de treino)", y=1.02)
plt.tight_layout()
plt.show()

## 7. Modelo e treinamento

**Fase 1:** o extrator ResNet-18 fica congelado; só o classificador linear é treinado.
**Fase 2:** o extrator é liberado e a rede inteira passa por fine-tune, com uma taxa de aprendizado menor.

O melhor modelo (maior acurácia de validação, entre as duas fases) é mantido ao final.

In [ ]:
model = DentalResNetTransfer(num_classes=NUM_CLASSES)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis (extrator congelado): {trainable_params:,} / {total_params:,}")

trainer = Trainer(model, device, head_lr=HEAD_LR, finetune_lr=FINETUNE_LR)
history = trainer.fit(
    train_loader,
    val_loader,
    frozen_epochs=FROZEN_EPOCHS,
    finetune_epochs=FINETUNE_EPOCHS,
)

## 8. Curvas de Loss e Acurácia

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

fase_transicao = history.phase.count("frozen")  # índice onde a Fase 2 (fine-tune) começa

ax1.plot(history.train_loss, label="Treino")
ax1.plot(history.val_loss, label="Validação")
ax1.axvline(fase_transicao - 0.5, color="gray", linestyle="--", label="Início do fine-tune")
ax1.set_xlabel("Época (contínua entre as duas fases)")
ax1.set_ylabel("Loss")
ax1.set_title("Curva de Loss")
ax1.legend()

ax2.plot(history.train_accuracy, label="Treino")
ax2.plot(history.val_accuracy, label="Validação")
ax2.axvline(fase_transicao - 0.5, color="gray", linestyle="--", label="Início do fine-tune")
ax2.set_xlabel("Época (contínua entre as duas fases)")
ax2.set_ylabel("Acurácia")
ax2.set_title("Curva de Acurácia")
ax2.legend()

plt.tight_layout()
plt.show()

## 9. Avaliação final e matriz de confusão

In [ ]:
test_loss, test_acc = trainer.evaluate(test_loader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}\n")

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        y_pred.extend(outputs.argmax(dim=1).cpu().tolist())
        y_true.extend(labels.tolist())

print(classification_report(y_true, y_pred, target_names=classes))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=classes, cmap="Blues", xticks_rotation=45, ax=ax,
)
ax.set_title(f"Matriz de Confusão — Teste (acurácia = {test_acc:.4f})")
plt.tight_layout()
plt.show()

## 10. Predição em uma imagem de exemplo

In [ ]:
classe_exemplo = random.choice(classes)
exemplo_path = random.choice(list((imagefolder_root / "test" / classe_exemplo).glob("*")))
pred = predict(exemplo_path, model, classes, IMAGE_SIZE, device)

plt.figure(figsize=(3, 3))
plt.imshow(Image.open(exemplo_path))
plt.title(f"Previsto: {pred.label}")
plt.axis('off')
plt.show()

print(f"Imagem          : {exemplo_path.name}")
print(f"Classe real     : {classe_exemplo}")
print(f"Classe prevista : {pred.label}")
print("Probabilidades  :")
for cls, prob in sorted(pred.probabilities.items(), key=lambda kv: -kv[1]):
    print(f"  {cls:<20} {prob:.1%}")

## 11. Salvar o modelo

A VM do Colab é descartada ao fim da sessão. Para não perder o checkpoint, **baixe-o pelo painel de arquivos** (ícone de pasta na barra lateral → `dental-image-classifier/artifacts/resnet18_transfer_model.pth` → ⋮ → Fazer download). Se o Drive estiver montado em `/content/drive`, a célula também copia o checkpoint para lá automaticamente.

In [ ]:
save_checkpoint(MODEL_OUT, model, classes, IMAGE_SIZE)
print(f"Modelo salvo em: {MODEL_OUT}")

# Copia para o Drive apenas se ele estiver de fato montado (drive.mount) —
# sem o mount, /content/drive seria só uma pasta local perdida ao fim da sessão.
drive_mount = Path("/content/drive/MyDrive")
if IN_COLAB and drive_mount.is_dir():
    DRIVE_OUT = drive_mount / "dental-image-classifier" / "artifacts"
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(MODEL_OUT, DRIVE_OUT / MODEL_OUT.name)
    print(f"Cópia no Drive : {DRIVE_OUT / MODEL_OUT.name}")
elif IN_COLAB:
    print("Drive não montado — baixe o checkpoint pelo painel de arquivos para não perdê-lo.")